In [ ]:
# NAME : KAUSHAL SHETE
# PRN : 20220802168
# BATCH : DS-1

# Import dependencies
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# Data Augmentation - Apply transformations
transform = transforms.Compose([
    transforms.RandomRotation(10),  # Rotate by ±10 degrees
    transforms.RandomHorizontalFlip(),  # Flip images horizontally
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize
])

# Load MNIST Dataset
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

# Split Training Data (80% train, 20% validation)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

# Dataloaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Define Neural Network with Dropout
class RegularizedNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)  # Dropout with 50% probability

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten image
        x = self.relu(self.fc1(x))
        x = self.dropout(x)  # Apply Dropout
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Initialize model, optimizer with L2 Regularization (weight_decay)
model = RegularizedNN()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # L2 Regularization
loss_fn = nn.CrossEntropyLoss()

# Early Stopping Setup
best_val_loss = float('inf')
patience = 5
counter = 0

# Training Loop with L1, L2 Regularization and Early Stopping
epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # L1 Regularization (Manually adding absolute sum of weights)
        l1_lambda = 1e-5
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        loss += l1_lambda * l1_norm

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation Step
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()

    # Early Stopping Check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    print(f"Epoch {epoch+1}: Train Loss={train_loss/len(train_loader):.4f}, Val Loss={val_loss/len(val_loader):.4f}")

# Testing the Model
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")



Epoch 1: Train Loss=0.7707, Val Loss=0.4282
Epoch 2: Train Loss=0.5638, Val Loss=0.3378
Epoch 3: Train Loss=0.5029, Val Loss=0.2621
Epoch 4: Train Loss=0.4699, Val Loss=0.2584
Epoch 5: Train Loss=0.4502, Val Loss=0.2261
Epoch 6: Train Loss=0.4316, Val Loss=0.2171
Epoch 7: Train Loss=0.4169, Val Loss=0.2049
Epoch 8: Train Loss=0.4110, Val Loss=0.2184
Epoch 9: Train Loss=0.4081, Val Loss=0.2010
Epoch 10: Train Loss=0.4024, Val Loss=0.1928
Epoch 11: Train Loss=0.4025, Val Loss=0.1980
Epoch 12: Train Loss=0.3919, Val Loss=0.1814
Epoch 13: Train Loss=0.3902, Val Loss=0.1822
Epoch 14: Train Loss=0.3851, Val Loss=0.1738
Epoch 15: Train Loss=0.3793, Val Loss=0.1771
Epoch 16: Train Loss=0.3860, Val Loss=0.1728
Epoch 17: Train Loss=0.3764, Val Loss=0.1739
Epoch 18: Train Loss=0.3773, Val Loss=0.1697
Epoch 19: Train Loss=0.3744, Val Loss=0.1665
Epoch 20: Train Loss=0.3730, Val Loss=0.1785
Test Accuracy: 90.65%
